# P10.6-AI — Notebook 59: entrenamiento foraminal Sagittal T1

Entrena un clasificador **2.5D multiclase** para estrechamiento foraminal neural izquierdo y derecho usando exclusivamente `train_manifest.csv` y `validation_manifest.csv` del Notebook 58.

El modelo comparte el backbone entre ambos lados e incorpora embeddings explícitos de **lado** y **nivel lumbar**. El `internal_test` permanece sellado para el Notebook 60.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


## Recursos y alcance

- Runtime recomendado: **Google Colab con GPU T4 o superior**.
- Autorizar Google Drive.
- No requiere token de GitHub.
- El token de Kaggle se solicita únicamente si las series Sagittal T1 necesarias no están disponibles en el runtime o en Drive.
- El notebook puede reanudarse: los crops `.npy` ya generados y el checkpoint se reutilizan.
- No abre ni carga `internal_test_manifest.csv`.


In [1]:
# 1) Dependencias mínimas
from __future__ import annotations
import importlib.util
import subprocess
import sys

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "kaggle": "kaggle",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *missing,
    ])
print({"installedNow": missing})


{'installedNow': ['pydicom']}


In [2]:
# 2) GPU y Google Drive
import getpass
import json
import os
from pathlib import Path

import torch
from google.colab import drive  # type: ignore

if not torch.cuda.is_available():
    raise RuntimeError(
        "Seleccioná GPU T4 o superior: Entorno de ejecución > Cambiar tipo de entorno."
    )

print({
    "gpu": torch.cuda.get_device_name(0),
    "gpuMemoryGiB": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ),
    "torch": torch.__version__,
})
drive.mount("/content/drive", force_remount=False)


{'gpu': 'Tesla T4', 'gpuMemoryGiB': 14.56, 'torch': '2.11.0+cu128'}
Mounted at /content/drive


In [3]:
# 3) Clonar o actualizar la rama e importar el pipeline
REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git",
        "clone",
        "--branch",
        REPO_REF,
        "--single-branch",
        REPO_URL,
        str(REPO_ROOT),
    ])
else:
    subprocess.check_call(
        ["git", "fetch", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "checkout", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "pull", "--ff-only", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()

sys.path.insert(0, str(REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_foraminal_training import (
    TrainConfig,
    build_cache,
    download_selected_series,
    find_data_root,
    load_manifests,
    prepare_samples,
    train_model,
)

print({"repoRef": REPO_REF, "repoSha": REPO_SHA})


{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': '2e249dfffd869e98bc9d3659a70b08298eb9af67'}


In [4]:
# 4) Rutas y configuración
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
RESULTS_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
SPLIT_ROOT = RESULTS_ROOT / "notebook58_foraminal_split"
RUN_ROOT = RESULTS_ROOT / "notebook59_foraminal_training"

MODEL_ROOT = (
    PFI_ROOT
    / "models"
    / "P10_6_rsna_findings"
    / "foraminal_sagittal_t1_2p5d"
)
CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"

LOCAL_DATA_ROOT = Path("/content/RSNA_LUMBAR_DISC")
DRIVE_DATA_ROOT = PFI_ROOT / "data" / "RSNA_LUMBAR_DISC"
CACHE_ROOT = Path("/content/rsna_foraminal_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"

CFG = TrainConfig(
    seed=2026,
    image_size=224,
    crop_size=256,
    batch_size=32,
    num_workers=2,
    max_epochs=15,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    model_name="efficientnet_b0",
    pretrained=True,
)

for path in (RUN_ROOT, MODEL_ROOT, CHECKPOINT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(CFG)
print({
    "splitRoot": str(SPLIT_ROOT),
    "runRoot": str(RUN_ROOT),
    "checkpointRoot": str(CHECKPOINT_ROOT),
    "cacheRoot": str(CACHE_ROOT),
})


TrainConfig(seed=2026, image_size=224, crop_size=256, batch_size=32, num_workers=2, max_epochs=15, patience=5, learning_rate=0.0002, weight_decay=0.0001, model_name='efficientnet_b0', pretrained=True, side_embedding_dim=8, level_embedding_dim=12, dropout=0.25, label_smoothing=0.05, severe_loss_multiplier=1.25, max_grad_norm=2.0, minimum_macro_f1=0.36, minimum_balanced_accuracy=0.45, minimum_severe_recall=0.3, minimum_moderate_recall=0.25)
{'splitRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook58_foraminal_split', 'runRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook59_foraminal_training', 'checkpointRoot': '/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings/foraminal_sagittal_t1_2p5d/checkpoints', 'cacheRoot': '/content/rsna_foraminal_cache'}


## Carga y auditoría de datos

La función siguiente verifica hashes, aprobación del Notebook 58, separación por estudio y presencia de las tres clases. Solo lee los manifests de entrenamiento y validación; del internal test únicamente comprueba que el archivo sellado exista.


In [5]:
# 5) Cargar train y validation sin acceder al internal test
train_manifest, validation_manifest, split_summary, manifest_hashes = (
    load_manifests(SPLIT_ROOT)
)

distribution = (
    train_manifest.groupby(["side", "level", "severity"])
    .size()
    .rename("train_rows")
    .reset_index()
)
validation_distribution = (
    validation_manifest.groupby(["side", "level", "severity"])
    .size()
    .rename("validation_rows")
    .reset_index()
)

print({
    "trainRows": len(train_manifest),
    "trainStudies": train_manifest["study_id"].nunique(),
    "validationRows": len(validation_manifest),
    "validationStudies": validation_manifest["study_id"].nunique(),
    "trainClassCounts": train_manifest["severity"].value_counts().to_dict(),
    "validationClassCounts": validation_manifest["severity"].value_counts().to_dict(),
    "internalTestAccessed": False,
    "officialTestAccessed": False,
})
display(distribution)
display(validation_distribution)


{'trainRows': 13774, 'trainStudies': 1380, 'validationRows': 2960, 'validationStudies': 296, 'trainClassCounts': {'normal_mild': 10739, 'moderate': 2496, 'severe': 539}, 'validationClassCounts': {'normal_mild': 2311, 'moderate': 534, 'severe': 115}, 'internalTestAccessed': False, 'officialTestAccessed': False}


,side,level,severity,train_rows
0,left,L1-L2,moderate,42
1,left,L1-L2,normal_mild,1337
2,left,L1-L2,severe,1
3,left,L2-L3,moderate,119
4,left,L2-L3,normal_mild,1254
5,left,L2-L3,severe,7
6,left,L3-L4,moderate,279
7,left,L3-L4,normal_mild,1074
8,left,L3-L4,severe,27
9,left,L4-L5,moderate,443


,side,level,severity,validation_rows
0,left,L1-L2,moderate,14
1,left,L1-L2,normal_mild,282
2,left,L2-L3,moderate,30
3,left,L2-L3,normal_mild,265
4,left,L2-L3,severe,1
5,left,L3-L4,moderate,73
6,left,L3-L4,normal_mild,217
7,left,L3-L4,severe,6
8,left,L4-L5,moderate,99
9,left,L4-L5,normal_mild,184


In [6]:
# 6) Resolver las series Sagittal T1 requeridas
DATA_ROOT, data_audits = find_data_root(
    [LOCAL_DATA_ROOT, DRIVE_DATA_ROOT],
    train_manifest,
    validation_manifest,
)

print({
    "audits": [
        {
            "root": audit.root,
            "complete": audit.complete,
            "requiredSeries": audit.required_series,
            "missingSeries": audit.missing_series,
            "missingExamples": list(audit.missing_examples[:5]),
        }
        for audit in data_audits
    ]
})

if DATA_ROOT is None:
    token = getpass.getpass(
        "Pegá tu KAGGLE_API_TOKEN para descargar solo las series requeridas: "
    ).strip()
    DATA_ROOT = download_selected_series(
        train_manifest,
        validation_manifest,
        LOCAL_DATA_ROOT,
        COMPETITION,
        token,
    )
    token = ""
    os.environ.pop("KAGGLE_API_TOKEN", None)

print({
    "dataRoot": str(DATA_ROOT),
    "sequence": "Sagittal T1",
    "internalTestAccessed": False,
})


{'audits': [{'root': '/content/RSNA_LUMBAR_DISC', 'complete': False, 'requiredSeries': 1681, 'missingSeries': 1681, 'missingExamples': ['100206310/2092806862', '1002894806/866293114', '1004726367/2526352865', '1008446160/2539455828', '1009445512/3088482668']}, {'root': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'complete': True, 'requiredSeries': 1681, 'missingSeries': 0, 'missingExamples': []}]}
{'dataRoot': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'sequence': 'Sagittal T1', 'internalTestAccessed': False}


## Preparación 2.5D

Cada muestra usa tres cortes sagitales adyacentes (`centro-1`, `centro`, `centro+1`) alrededor de la coordenada foraminal. Se recorta una región de 256×256 y se redimensiona a 224×224. No se aplica espejo horizontal porque invertiría el eje anteroposterior.


In [7]:
# 7) Preparar muestras y construir/reutilizar caché local
train_samples = prepare_samples(train_manifest, DATA_ROOT, "train")
validation_samples = prepare_samples(
    validation_manifest,
    DATA_ROOT,
    "validation",
)

train_cache_audit = build_cache(
    train_samples,
    CACHE_ROOT,
    "train",
    CFG,
)
validation_cache_audit = build_cache(
    validation_samples,
    CACHE_ROOT,
    "validation",
    CFG,
)

print({
    "trainCache": train_cache_audit,
    "validationCache": validation_cache_audit,
    "internalTestAccessed": False,
})


cache train por serie:   0%|          | 0/1385 [00:00<?, ?it/s]

cache validation por serie:   0%|          | 0/296 [00:00<?, ?it/s]

{'trainCache': {'split': 'train', 'expectedSamples': 13774, 'builtSamples': 13774, 'reusedSamples': 0, 'cacheFiles': 13774, 'minutes': 83.71}, 'validationCache': {'split': 'validation', 'expectedSamples': 2960, 'builtSamples': 2960, 'reusedSamples': 0, 'cacheFiles': 2960, 'minutes': 17.99}, 'internalTestAccessed': False}


## Entrenamiento

El muestreo pondera simultáneamente la clase y el estrato `lado × nivel × severidad`. La selección del checkpoint combina macro F1, balanced accuracy, recall de `Severe` y recall de `Moderate`. El entrenamiento usa AMP, AdamW, reducción de learning rate y early stopping.


In [8]:
# 8) Entrenar y seleccionar el checkpoint usando validation
torch.cuda.empty_cache()

training_summary = train_model(
    train_samples,
    validation_samples,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
    RUN_ROOT,
    manifest_hashes,
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
    config=CFG,
)

print(json.dumps({
    "status": training_summary["status"],
    "approved": training_summary["approved"],
    "nextNotebook": training_summary["nextNotebook"],
    "bestEpoch": training_summary["bestEpoch"],
    "bestSelectionScore": training_summary["bestSelectionScore"],
    "validationMetrics": training_summary["validationMetrics"],
    "processGates": training_summary["processGates"],
    "metricGates": training_summary["metricGates"],
    "checkpoint": training_summary["checkpoint"],
    "internalTestAccessed": False,
}, indent=2, ensure_ascii=False))


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

train:   0%|          | 0/431 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


validation:   0%|          | 0/93 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


{'epoch': 9, 'learning_rate': 5e-05, 'train_macro_f1': 0.9866049181566042, 'train_balanced_accuracy': 0.9886903297695852, 'train_normal_mild_recall': 0.9785767790262172, 'train_moderate_recall': 0.9874942102825383, 'train_severe_recall': 1.0, 'train_weighted_log_loss': 0.1510503143498288, 'train_selection_score': 0.9905639707332918, 'train_loss': 0.2187002324886621, 'validation_macro_f1': 0.607643038605289, 'validation_balanced_accuracy': 0.581804734280257, 'validation_normal_mild_recall': 0.9251406317611424, 'validation_moderate_recall': 0.4550561797752809, 'validation_severe_recall': 0.3652173913043478, 'validation_weighted_log_loss': 0.5829398028481982, 'validation_selection_score': 0.5253183648157949, 'validation_loss': 0.9931401652258796}
{'earlyStopping': True, 'epoch': 9}


validation:   0%|          | 0/93 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: T

{
  "status": "APPROVED_FOR_NOTEBOOK_60",
  "approved": true,
  "nextNotebook": 60,
  "bestEpoch": 4,
  "bestSelectionScore": 0.5605953661265175,
  "validationMetrics": {
    "macro_f1": 0.5951860462078994,
    "balanced_accuracy": 0.6194763976406836,
    "normal_mild_recall": 0.87321505841627,
    "moderate_recall": 0.5243445692883895,
    "severe_recall": 0.4608695652173913,
    "weighted_log_loss": 0.6164788508082008,
    "selection_score": 0.5605953661265175,
    "loss": 0.973122222359116
  },
  "processGates": {
    "sourceNotebook58Approved": true,
    "trainValidationStudyIsolation": true,
    "trainContainsAllClasses": true,
    "validationContainsAllClasses": true,
    "bestCheckpointExists": true,
    "finiteValidationMetrics": true,
    "internalTestNotAccessed": true,
    "officialTestNotAccessed": true,
    "humanReviewRequired": true,
    "notClinicalDiagnosis": true
  },
  "metricGates": {
    "macroF1": true,
    "balancedAccuracy": true,
    "severeRecall": true,
    "

In [9]:
# 9) Gate final y evidencia generada
required_outputs = [
    "training_history.csv",
    "validation_predictions.csv",
    "validation_metrics_by_group.csv",
    "sampling_audit.json",
    "model_card.md",
    "training_summary.json",
]
missing_outputs = [
    name for name in required_outputs
    if not (RUN_ROOT / name).is_file()
]

if missing_outputs:
    raise RuntimeError(f"Faltan outputs del Notebook 59: {missing_outputs}")
if not (CHECKPOINT_ROOT / "best_checkpoint.pt").is_file():
    raise RuntimeError("No se generó best_checkpoint.pt.")

print({
    "status": training_summary["status"],
    "outputs": required_outputs,
    "bestCheckpoint": str(CHECKPOINT_ROOT / "best_checkpoint.pt"),
    "internalTestSealedUntilNotebook60": True,
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
})

if not training_summary["approved"]:
    raise RuntimeError(
        "El entrenamiento terminó, pero no superó todos los gates de validación. "
        "Revisar training_summary.json antes del Notebook 60."
    )


{'status': 'APPROVED_FOR_NOTEBOOK_60', 'outputs': ['training_history.csv', 'validation_predictions.csv', 'validation_metrics_by_group.csv', 'sampling_audit.json', 'model_card.md', 'training_summary.json'], 'bestCheckpoint': '/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings/foraminal_sagittal_t1_2p5d/checkpoints/best_checkpoint.pt', 'internalTestSealedUntilNotebook60': True, 'officialTestAccessed': False, 'humanReviewRequired': True, 'notClinicalDiagnosis': True}


## Conclusión esperada

Una ejecución aprobada termina con `APPROVED_FOR_NOTEBOOK_60`. El checkpoint seleccionado y sus métricas quedan congelados en Drive. Recién el Notebook 60 podrá abrir el `internal_test_manifest.csv` para realizar una única evaluación final y decidir la exportación del modelo.
